In [1]:
import numpy as np
import pandas as pd
import duckdb
import pyfixest as pf

In [39]:
sensor_data = pd.read_parquet("/scicore/home/meiera/schulz0022/projects/river-pollution-brazil/data/sensor_data/water_quality_assembled.parquet").reset_index()
sensor_data["year"] = sensor_data.datetime.dt.year
sensor_data

,station_code,datetime,rained,depth,air_temperature,sample_temperature,ph,color,turbidity,electrical_conductivity,...,date,trench_id,streamflow_discharge_day,streamflow_discharge_mean_7d,streamflow_discharge_mean_31d,streamflow_match_count,streamflow_nonnull_day_count,streamflow_total_weight,streamflow_nearest_distance_m,year
0,10100000,1996-07-01 10:00:00,0.0,NaN,NaN,27.600000,7.60,NaN,NaN,10.700000,...,1996-07-01,25130,39724.773704,42490.873791,54242.727891,3,3,2.975611,0.0,1996.0
1,10100000,1996-09-11 16:00:00,0.0,NaN,NaN,25.200001,7.70,NaN,29.000000,203.000000,...,1996-09-11,25130,21815.145252,22008.451241,25836.115092,3,3,2.975611,0.0,1996.0
2,10100000,1996-12-21 10:55:00,0.0,NaN,NaN,NaN,7.80,NaN,20.000000,164.800003,...,1996-12-21,25130,43343.318655,40474.179104,34872.594574,3,3,2.975611,0.0,1996.0
3,10100000,1997-03-19 09:00:00,0.0,NaN,NaN,27.799999,7.00,NaN,29.000000,159.300003,...,1997-03-19,25130,62757.391554,62046.488915,55081.824021,3,3,2.975611,0.0,1997.0
4,10100000,1997-06-06 15:00:00,0.0,0.0,NaN,28.000000,6.60,NaN,22.000000,108.900002,...,1997-06-06,25130,63925.379606,64086.875347,64132.666797,3,3,2.975611,0.0,1997.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208607,88850000,2023-09-06 12:00:00,0.0,0.5,16.0,11.300000,6.92,NaN,49.180000,NaN,...,2023-09-06,70632,NaN,NaN,NaN,0,0,0.000000,NaN,2023.0
208608,88850000,2023-11-17 15:00:00,0.0,0.5,28.0,19.900000,7.24,NaN,NaN,NaN,...,2023-11-17,70632,NaN,NaN,NaN,0,0,0.000000,NaN,2023.0
208609,88850000,2024-02-29 10:30:00,0.0,0.5,26.0,23.299999,7.23,NaN,19.219999,NaN,...,2024-02-29,70632,2.146300,2.738414,4.022665,1,1,1.000000,0.0,2024.0
208610,88850000,2024-07-25 10:45:00,0.0,0.5,19.0,14.900000,7.10,NaN,10.700000,NaN,...,2024-07-25,70632,3.725500,3.895843,6.130013,1,1,1.000000,0.0,2024.0


In [38]:
import duckdb
import numpy as np
import pandas as pd

LC_PATH = "/scicore/home/meiera/schulz0022/projects/river-pollution-brazil/data/land_cover/land_cover_assembled_sensor.parquet"

PSEUDOCOUNT = 1e-4

# 25km-wide rings out to 500km, with an open-ended tail beyond that. Generated
# rather than hardcoded so it stays correct if the ring width/extent changes
# upstream, and so any bucket value missing from this map is immediately
# visible (see sanity check 2) instead of silently dropped by the join.
_RING_WIDTH_KM = 25
BUCKET_MAP = {
    b: (f"{b}_{b + _RING_WIDTH_KM}km", b + _RING_WIDTH_KM / 2)
    for b in range(0, 500, _RING_WIDTH_KM)
}
BUCKET_MAP[500] = ("500km_plus", 750.0)  # open-ended tail beyond 500km

# Leaf-level composition after resolving the c3/c4 parent-child mismatches.
LEAF_CLASSES = [
    "forest", "nonforest_nat", "pasture", "agriculture",
    "farming_unclassified", "urban", "mining", "other", "water",
]
ALR_CLASSES = ["pasture", "agriculture", "farming_unclassified", "urban", "mining", "other"]


def build_bucket_map(con: duckdb.DuckDBPyConnection) -> None:
    bucket_rows = ", ".join(
        f"({bucket}, '{label}', {midpoint})"
        for bucket, (label, midpoint) in BUCKET_MAP.items()
    )
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE bucket_map AS
        SELECT * FROM (VALUES {bucket_rows}) AS t(bucket, bucket_label, midpoint_km)
    """)


def sanity_check_n_vs_share(con: duckdb.DuckDBPyConnection, table: str = "lc_long") -> None:
    """
    share IS NULL is the ground-truth signal that a bucket has no valid area.
    n=0 was assumed to be an equivalent proxy for that but isn't reliable in
    this data -- the transform keys off `share IS NULL` directly, so this
    check is informational only, not a precondition.
    """
    mismatches = con.execute(f"""
        SELECT SUM(((n = 0) <> (share IS NULL))::INT) FROM {table}
    """).fetchone()[0]
    if mismatches:
        print(f"NOTE: {mismatches} rows have n=0 disagreeing with share IS NULL "
              f"(informational -- transform does not rely on n)")


def transform(con: duckdb.DuckDBPyConnection) -> duckdb.DuckDBPyRelation:
    unpivot_sql = "\n            UNION ALL\n            ".join(
        f"SELECT station_code, year, bucket, '{c}' AS class_short, {c} AS share FROM bucket_resolved"
        for c in LEAF_CLASSES
    )
    wide_cols_sql = ",\n                ".join(
        f"MAX(dw_share) FILTER (WHERE class_short = '{c}') AS lc_{c}"
        for c in LEAF_CLASSES
    )
    alr_cols_sql = ", ".join(
        f"LN((lc_{c} + {PSEUDOCOUNT}) / (lc_forest + lc_nonforest_nat + {PSEUDOCOUNT})) AS alr_{c}"
        for c in ALR_CLASSES
    )
    select_lc_cols = ", ".join(f"lc_{c}" for c in LEAF_CLASSES)

    return con.sql(f"""
        WITH bucket_pivot AS (
            SELECT
                station_code, year, bucket,
                MAX(share) FILTER (WHERE land_cover_class = 1)  AS forest,
                MAX(share) FILTER (WHERE land_cover_class = 2)  AS nonforest_nat,
                MAX(share) FILTER (WHERE land_cover_class = 3)  AS c3,
                MAX(share) FILTER (WHERE land_cover_class = 30) AS pasture,
                MAX(share) FILTER (WHERE land_cover_class = 31) AS agriculture,
                MAX(share) FILTER (WHERE land_cover_class = 4)  AS c4,
                MAX(share) FILTER (WHERE land_cover_class = 40) AS urban,
                MAX(share) FILTER (WHERE land_cover_class = 41) AS mining,
                MAX(share) FILTER (WHERE land_cover_class = 42) AS other_raw,
                MAX(share) FILTER (WHERE land_cover_class = 5)  AS water
            FROM lc_long
            GROUP BY 1, 2, 3
        ),
        -- resolve c3 -> pasture/agriculture and c4 -> urban/mining/other
        -- mismatches by keeping unattributed mass explicit instead of
        -- dropping it (previous version) or guessing a split
        bucket_resolved AS (
            SELECT
                station_code, year, bucket,
                forest, nonforest_nat, pasture, agriculture,
                CASE WHEN c3 IS NULL THEN NULL
                     ELSE GREATEST(c3 - COALESCE(pasture, 0) - COALESCE(agriculture, 0), 0)
                END AS farming_unclassified,
                urban, mining,
                CASE WHEN c4 IS NULL THEN NULL
                     ELSE COALESCE(other_raw, 0)
                          + GREATEST(COALESCE(c4, 0) - COALESCE(urban, 0) - COALESCE(mining, 0) - COALESCE(other_raw, 0), 0)
                END AS other,
                water
            FROM bucket_pivot
        ),
        leaf AS (
            {unpivot_sql}
        ),
        -- total leaf-class share per station/year/bucket (< 1 to the extent
        -- c0/no-data pixels exist); NULL if the bucket has no valid area at all
        leaf_totals AS (
            SELECT station_code, year, bucket, SUM(share) AS leaf_total
            FROM leaf GROUP BY 1, 2, 3
        ),
        leaf_renorm AS (
            SELECT l.station_code, l.year, l.bucket, l.class_short,
                   CASE WHEN l.share IS NULL THEN NULL
                        ELSE l.share / NULLIF(t.leaf_total, 0) END AS share_renorm
            FROM leaf l JOIN leaf_totals t USING (station_code, year, bucket)
        ),
        bucket_avail AS (
            SELECT lt.station_code, lt.year, lt.bucket, bm.midpoint_km, lt.leaf_total
            FROM leaf_totals lt
            JOIN bucket_map bm USING (bucket)
        ),
        bucket_weighted AS (
            SELECT *, CASE WHEN leaf_total IS NOT NULL AND leaf_total > 0
                          THEN 1.0 / SQRT(midpoint_km) ELSE 0 END AS raw_weight
            FROM bucket_avail
        ),
        -- renormalize weights over only the buckets that exist for this station
        -- (small headwater catchments won't have a 250-500km or 500km+ ring)
        bucket_weight_norm AS (
            SELECT station_code, year, bucket,
                   raw_weight / NULLIF(SUM(raw_weight) OVER (PARTITION BY station_code, year), 0) AS weight
            FROM bucket_weighted
        ),
        weighted_class AS (
            SELECT lr.station_code, lr.year, lr.class_short,
                   SUM(lr.share_renorm * w.weight) AS dw_share
            FROM leaf_renorm lr
            JOIN bucket_weight_norm w USING (station_code, year, bucket)
            WHERE lr.share_renorm IS NOT NULL
            GROUP BY 1, 2, 3
        ),
        wide AS (
            SELECT station_code, year,
                {wide_cols_sql}
            FROM weighted_class GROUP BY station_code, year
        )
        SELECT station_code, year,
               {select_lc_cols},
               (lc_forest + lc_nonforest_nat) AS lc_nat,
               {alr_cols_sql}
        FROM wide
        ORDER BY station_code, year
    """)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW lc_long AS SELECT * FROM read_parquet('{LC_PATH}')")

sanity_check_n_vs_share(con)
build_bucket_map(con)

land_cover = transform(con).df()

# index on an actual datetime (Jan 1 of each land-cover year) rather than a
# bare int year, so this lines up directly against date-indexed sensor data.
# NB: sensor readings won't fall exactly on Jan 1 -- use pd.merge_asof
# (by=station_code, direction="backward") to attach the right year to each reading.
land_cover

NOTE: 8204 rows have n=0 disagreeing with share IS NULL (informational -- transform does not rely on n)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,station_code,year,lc_forest,lc_nonforest_nat,lc_pasture,lc_agriculture,lc_farming_unclassified,lc_urban,lc_mining,lc_other,lc_water,lc_nat,alr_pasture,alr_agriculture,alr_farming_unclassified,alr_urban,alr_mining,alr_other
0,10100000,1996,0.933115,0.000405,0.002125,0.000000,0.000000,0.002547,0.0,0.003068,0.058740,0.933520,-6.039272,-9.141654,-9.141654,-5.865722,-9.141654,-5.685813
1,10100000,1997,0.933913,0.000484,0.001923,0.000000,0.000000,0.002560,0.0,0.003147,0.057973,0.934397,-6.135367,-9.142593,-9.142593,-5.861704,-9.142593,-5.662325
2,10100000,1998,0.932905,0.000414,0.002131,0.000000,0.000000,0.002564,0.0,0.003318,0.058668,0.933319,-6.036437,-9.141440,-9.141440,-5.859077,-9.141440,-5.609861
3,10200000,1998,0.993710,0.000015,0.000081,0.000000,0.000000,0.000000,0.0,0.000000,0.006194,0.993725,-8.613445,-9.204146,-9.204146,-9.204146,-9.204146,-9.204146
4,10200000,1999,0.993664,0.000016,0.000081,0.000000,0.000000,0.000000,0.0,0.000000,0.006239,0.993680,-8.610386,-9.204101,-9.204101,-9.204101,-9.204101,-9.204101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58293,88850000,2019,0.367578,0.278671,0.000000,0.094536,0.254678,0.001893,0.0,0.002389,0.000255,0.646249,-8.773925,-1.921299,-0.930947,-5.781646,-8.773925,-5.559650
58294,88850000,2020,0.366446,0.258539,0.000000,0.094248,0.276131,0.002056,0.0,0.002339,0.000241,0.624985,-8.740473,-1.890893,-0.816652,-5.669448,-8.740473,-5.546420
58295,88850000,2022,0.349702,0.219977,0.000000,0.094893,0.330632,0.002328,0.0,0.002151,0.000318,0.569678,-8.647832,-1.791441,-0.543939,-5.458330,-8.647832,-5.534094
58296,88850000,2023,0.339334,0.201311,0.000000,0.094874,0.359698,0.002508,0.0,0.001918,0.000357,0.540645,-8.595533,-1.739345,-0.407404,-5.334543,-8.595533,-5.590833


In [40]:
merged = pd.merge(
    sensor_data,
    land_cover.reset_index(),
    on = ["station_code", "year"]
)
merged

,station_code,datetime,rained,depth,air_temperature,sample_temperature,ph,color,turbidity,electrical_conductivity,...,lc_mining,lc_other,lc_water,lc_nat,alr_pasture,alr_agriculture,alr_farming_unclassified,alr_urban,alr_mining,alr_other
0,10100000,1996-07-01 10:00:00,0.0,NaN,NaN,27.600000,7.60,NaN,NaN,10.700000,...,0.0,0.003068,0.058740,0.933520,-6.039272,-9.141654,-9.141654,-5.865722,-9.141654,-5.685813
1,10100000,1996-09-11 16:00:00,0.0,NaN,NaN,25.200001,7.70,NaN,29.000000,203.000000,...,0.0,0.003068,0.058740,0.933520,-6.039272,-9.141654,-9.141654,-5.865722,-9.141654,-5.685813
2,10100000,1996-12-21 10:55:00,0.0,NaN,NaN,NaN,7.80,NaN,20.000000,164.800003,...,0.0,0.003068,0.058740,0.933520,-6.039272,-9.141654,-9.141654,-5.865722,-9.141654,-5.685813
3,10100000,1997-03-19 09:00:00,0.0,NaN,NaN,27.799999,7.00,NaN,29.000000,159.300003,...,0.0,0.003147,0.057973,0.934397,-6.135367,-9.142593,-9.142593,-5.861704,-9.142593,-5.662325
4,10100000,1997-06-06 15:00:00,0.0,0.0,NaN,28.000000,6.60,NaN,22.000000,108.900002,...,0.0,0.003147,0.057973,0.934397,-6.135367,-9.142593,-9.142593,-5.861704,-9.142593,-5.662325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183529,88850000,2023-09-06 12:00:00,0.0,0.5,16.0,11.300000,6.92,NaN,49.180000,NaN,...,0.0,0.001918,0.000357,0.540645,-8.595533,-1.739345,-0.407404,-5.334543,-8.595533,-5.590833
183530,88850000,2023-11-17 15:00:00,0.0,0.5,28.0,19.900000,7.24,NaN,NaN,NaN,...,0.0,0.001918,0.000357,0.540645,-8.595533,-1.739345,-0.407404,-5.334543,-8.595533,-5.590833
183531,88850000,2024-02-29 10:30:00,0.0,0.5,26.0,23.299999,7.23,NaN,19.219999,NaN,...,0.0,0.001932,0.000340,0.531730,-8.578910,-1.721848,-0.366658,-5.298994,-8.578910,-5.567376
183532,88850000,2024-07-25 10:45:00,0.0,0.5,19.0,14.900000,7.10,NaN,10.700000,NaN,...,0.0,0.001932,0.000340,0.531730,-8.578910,-1.721848,-0.366658,-5.298994,-8.578910,-5.567376


In [56]:
merged.columns[100:150]

Index(['phosdrin', 'ddepp', 'ethyl_azinphos', 'diazinon', 'fecal_streptococci',
       'salmonella', 'coliphages', 'heterotrophic_bacteria', 'protozoa',
       'fungi', 'algae', 'plate_bacteria_count', 'chlorophyll',
       'oils_and_grease', 'total_alkalinity', 'total_organic_carbon',
       'hydrocarbons', 'total_orthophosphate', 'total_chromium',
       'methyl_parathion', 'organic_nitrogen', 'total_sodium',
       'total_magnesium', 'dissolved_silica', 'total_potassium',
       'total_calcium', 'total_iron', 'liquid_discharge', 'total_phosphorus',
       'total_bismuth', 'acidity', 'total_kjeldahl_nitrogen',
       'albuminoid_nitrogen', 'transparency', 'enteropathogenic_bacteria',
       'total_zooplankton', 'ammonia', 'water_quality_index', 'updated_at',
       'created_at', 'updated_by', 'thermotolerant_coliforms', 'escherichia',
       'dissolved_aluminum', 'dissolved_boron', 'free_cyanide',
       'dissolved_copper', 'specific_conductivity', 'cyanobacteria_density',
       'ma

In [57]:
fit = pf.feols(
    "biochemical_oxygen_demand ~ alr_pasture * streamflow_discharge_day | station_code + year",
    vcov = {"CRV1": "station_code"},
    data = merged
)

fit.summary()

###

Estimation:  OLS
Dep. var.: biochemical_oxygen_demand, Fixed effects: station_code + year
sample: None = all
Inference:  CRV1
Observations:  81119

| Coefficient                          |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| alr_pasture                          |      0.476 |        0.417 |     1.141 |      0.254 | -0.342 |   1.293 |
| streamflow_discharge_day             |     -0.000 |        0.000 |    -2.173 |      0.030 | -0.000 |  -0.000 |
| alr_pasture:streamflow_discharge_day |     -0.000 |        0.000 |    -2.081 |      0.038 | -0.000 |  -0.000 |
---
RMSE: 61.227 R2: 0.091 R2 Within: 0.0 


/scicore/home/meiera/schulz0022/miniforge-pypy3/envs/311/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 166 singleton fixed effect(s) dropped from the model.
  warnings.warn(
